In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Load ---
ret = pd.read_parquet("data/processed/returns_linear.parquet")
bench = pd.read_parquet("data/processed/benchmark_ret.parquet")["SPY"]
weights = pd.read_parquet("data/portfolio_weights/ew_monthly_wide.parquet")

# --- Portfolio return ---
w_prev = weights.shift(1).reindex(ret.index).ffill()
common = ret.columns.intersection(weights.columns)
ret_p = (w_prev[common] * ret[common]).sum(axis=1)
nav_p = (1 + ret_p.fillna(0)).cumprod()
drawdown = nav_p / nav_p.cummax() - 1

# --- Rolling metrics ---
sharpe_252 = (ret_p.rolling(252).mean() / ret_p.rolling(252).std()) * np.sqrt(252)
vol_20 = ret_p.rolling(20).std() * np.sqrt(252)
te_252 = (ret_p - bench).rolling(252).std() * np.sqrt(252)

# --- Combine ---
perf = pd.DataFrame({
    "ret_p": ret_p,
    "NAV": nav_p,
    "drawdown": drawdown,
    "Sharpe_252": sharpe_252,
    "VolAnn_20": vol_20,
    "TE_252": te_252
})
Path("data/analytics").mkdir(parents=True, exist_ok=True)
perf.to_parquet("data/analytics/performance_metrics.parquet")

perf.tail()

,ret_p,NAV,drawdown,Sharpe_252,VolAnn_20,TE_252
2024-12-24,0.012407,5.177790,-0.004842,2.748470,0.126428,0.050180
2024-12-25,0.000000,5.177790,-0.004842,2.651331,0.125691,0.050180
2024-12-26,-0.001511,5.169966,-0.006346,2.639851,0.125925,0.050205
2024-12-27,-0.012116,5.107329,-0.018384,2.490610,0.131488,0.050240
2024-12-30,-0.009975,5.056382,-0.028176,2.430299,0.134944,0.050159
